# Tract multi-group controls

Port of `packages/niivue/examples/tract.groups.html`. Selects multiple named TRX groups and colors each selected group with a fixed palette.


In [ ]:
import ipywidgets as widgets
from IPython.display import display
from ipyniivue import NiiVue

BASE_URL = "https://niivue.github.io/mono"
VOLUMES = f"{BASE_URL}/volumes"
MESHES = f"{BASE_URL}/meshes"

group_names = [
    "AC",
    "AF_L",
    "AF_R",
    "CC",
    "CST_L",
    "CST_R",
    "IFOF_L",
    "IFOF_R",
    "ILF_L",
    "ILF_R",
    "SLF1_L",
    "SLF1_R",
    "SLF2_L",
    "SLF2_R",
    "TR_S_L",
    "TR_S_R",
    "UF_L",
    "UF_R",
    "VOF_L",
    "VOF_R"
]

group_palette = [
    [0, 255, 0, 255],
    [255, 0, 0, 255],
    [25, 25, 255, 255],
    [255, 180, 0, 255],
    [180, 0, 255, 255],
    [0, 180, 255, 255],
]

nv = NiiVue(background_color=[1, 1, 1, 1], slice_type=4, volume_illumination=0.5, backend="webgl2")

slice_type = widgets.Dropdown(options=[("Render", 4), ("A+C+S+R", 3)], value=4, description="Slice")
selected_groups = widgets.SelectMultiple(options=group_names, value=("AC", "AF_L", "AF_R"), rows=8, description="Groups")
color_mode = widgets.Dropdown(options=[("Selected groups", "groups"), ("Local direction", "local"), ("Global direction", "global"), ("Fixed", "fixed")], value="groups", description="Color")
decimation = widgets.IntSlider(value=1, min=1, max=20, step=1, description="Decimation")
xray = widgets.Checkbox(value=False, description="XRay")

def update_slice(change):
    nv.slice_type = change["new"]

def update_decimation(change):
    nv.set_tract_options(0, {"decimation": change["new"]})

def update_xray(change):
    nv.mesh_x_ray = 0.005 if change["new"] else 0.0

def apply_color(change=None):
    if color_mode.value == "groups":
        group_colors = {
            name: group_palette[index % len(group_palette)]
            for index, name in enumerate(selected_groups.value)
        }
        nv.set_tract_options(0, {"groupColors": group_colors, "colorBy": "fixed"})
    else:
        nv.set_tract_options(0, {"groupColors": None, "colorBy": color_mode.value})

slice_type.observe(update_slice, names="value")
decimation.observe(update_decimation, names="value")
xray.observe(update_xray, names="value")
selected_groups.observe(apply_color, names="value")
color_mode.observe(apply_color, names="value")

controls = widgets.VBox([
    widgets.HBox([slice_type, color_mode, xray]),
    widgets.HBox([selected_groups, decimation]),
])
display(controls)
display(nv)

nv.set_clip_planes([[0.1, 180, 20], [0.1, 0, -20]])
nv.load_volumes([{"url": f"{VOLUMES}/mni152.nii.gz"}])
nv.load_meshes([{ "url": f"{MESHES}/yeh2022.trx", "rgba255": [0, 142, 200, 255] }])
apply_color()
